In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import KFold
import joblib
import time

In [2]:
df_train = pd.read_csv("Data/encode_train.csv")
df_valid = pd.read_csv("Data/encode_valid.csv")
df_test = pd.read_csv("Data/encode_test.csv")

In [3]:
bool_cols = [
    'FirstTimeHomebuyerFlag',
    'SuperConformingFlag',
    'CreditScore_MissFLag',
    'OriginalDTI_MissFLag',
    'HighRiskCredit',
    'HighLTV',
    'HighDTI',
    'HighInterestRate',
    'LTV_MissFlag'
]

cat_cols = [
    'NumberOfUnits',
    'OccupancyStatus',
    'Channel',
    'PropertyType',
    'LoanPurpose',
    'ProgramIndicator',
    'PropertyValMethod',
    'BalloonIndicator'
]

num_cols = [
    'CreditScore',
    'MI_Pct',
    'OriginalDTI',
    'OriginalUPB',
    'OriginalLTV',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    '0_1_UPB_Diff',
    '1_2_UPB_Diff',
    '2_3_UPB_Diff',
    '3_4_UPB_Diff',
    '4_5_UPB_Diff',
    '5_6_UPB_Diff',
    '6_7_UPB_Diff',
    '7_8_UPB_Diff',
    '8_9_UPB_Diff',
    '9_10_UPB_Diff',
    '10_11_UPB_Diff',
    '11_12_UPB_Diff',
    '12_13_UPB_Diff',
    'longest_unchanged_UPB',
    'avg_repayment_ratio',
    'pct_months_late',
    'Principal_reduction_rate',
    'Amortization_slope',
    'UPB_rebound_count',
    'UPB_autocorr',
    'UPB_skew',
    'UPB_kurtosis',
    'LTV_mean',
    'LTV_std',
    'LTV_slope',
    'LTV_rebound_count',
    'Interest_rate_std',
    'Interest_rate_slope',
    'Early_repayment_ratio',
    'CreditScore_LTV_Ratio',
    'CreditScore_DTI_Ratio',
    'LTV_DTI_Product',
    'DebtServiceRatio',
    'CompositeRiskScore',
    'OriginalLTV_delta',
    'MSA_freq_enc',
    'PropertyState_freq_enc',
    'SellerName_freq_enc',
    'ServicerName_freq_enc'
]

In [4]:
features = bool_cols + cat_cols + num_cols

X_train = df_train[features]
X_valid = df_valid[features]
X_test = df_test[features]

y_valid = df_valid[["index", "target"]]

In [5]:
def adjusted_gower_normalization(residuals, X_ref, feature, num_cols, cat_cols, bool_cols):
    if feature in num_cols:
        rng = np.nanmax(X_ref[feature]) - np.nanmin(X_ref[feature])
        rng = rng if rng > 1e-6 else np.std(X_ref[feature]) + 1e-6
        return residuals / rng
    elif feature in cat_cols or feature in bool_cols:
        return np.clip(residuals, 0, 1)
    else:
        return residuals

In [11]:
class RFOD:
    def __init__(self,
                 n_estimators=200,
                 n_splits=5,
                 feature_frac=0.7,
                 patience=5,
                 n_jobs=-1,
                 agg_weight_method="adaptive",
                 random_state=42,
                 verbose=True):
        self.n_estimators = n_estimators
        self.n_splits = n_splits
        self.feature_frac = feature_frac
        self.patience = patience
        self.n_jobs = n_jobs
        self.agg_weight_method = agg_weight_method
        self.random_state = random_state
        self.verbose = verbose

        # Attributes to be set after fitting
        self.selected_features_ = None
        self.feature_ranking_ = None
        self.preprocessor_ = None
        self.feature_indices_ = None
        self.final_AP_ = None
        self.ROC_AUC_ = None

    # ---------------------------
    # Internal utility: uncertainty weighting
    # ---------------------------
    def _weight_from_uncertainty(self, uncert, residual_mean=None, eps=1e-8):
        method = self.agg_weight_method
        if method == "inv":
            return 1.0 / (uncert + eps)
        elif method == "logodds":
            u = np.clip(uncert, eps, 1.0 - eps)
            return np.log((1.0 - u) / (u + eps))
        elif method == "adaptive":
            if residual_mean is None:
                residual_mean = np.mean(uncert, axis=0)
            return 1.0 / (uncert + 0.5 * residual_mean + eps)
        return np.ones_like(uncert)

    # ---------------------------
    # Internal: train one feature
    # ---------------------------
    def _train_one_feature(self, i, feat, Xtr_proc, Xval_proc, feature_indices, all_features,
                           num_cols, cat_cols, bool_cols, y_valid, X_train_df,
                           progress_bar=None):

        rng = np.random.default_rng(self.random_state + i)
        others = [f for f in all_features if f != feat]
        selected_features = rng.choice(others, size=max(1, int(len(others) * self.feature_frac)), replace=False)
        idxs_sel = [feature_indices[f] for f in selected_features]

        Xtr_sel = Xtr_proc[:, idxs_sel]
        Xval_sel = Xval_proc[:, idxs_sel]
        ytr_full = Xtr_proc[:, feature_indices[feat]]
        yval_full = Xval_proc[:, feature_indices[feat]]

        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        preds_val = np.zeros(len(yval_full))
        uncert_val = np.zeros(len(yval_full))

        for tr_idx, val_idx in kf.split(Xtr_sel):
            Xtr, Xval = Xtr_sel[tr_idx], Xtr_sel[val_idx]
            ytr = ytr_full[tr_idx]

            if feat in cat_cols or feat in bool_cols:
                model = RandomForestClassifier(n_estimators=self.n_estimators,
                                               random_state=self.random_state, n_jobs=1)
                ytr_ = np.round(ytr).astype(int)
                model.fit(Xtr, ytr_)
                prob = model.predict_proba(Xval_sel)
                prob_max = np.max(prob, axis=1)
                prob_true = np.array([
                    prob[i, np.where(model.classes_ == np.round(yval_full[i]).astype(int))[0][0]]
                    if np.round(yval_full[i]).astype(int) in model.classes_ else 0
                    for i in range(len(yval_full))
                ])
                score_fold = 1.0 - prob_true
                uncert_fold = 1.0 - prob_max
            else:
                model = RandomForestRegressor(n_estimators=self.n_estimators,
                                              max_depth=6,
                                              min_samples_leaf=5,
                                              min_samples_split=10,
                                              random_state=self.random_state,
                                              n_jobs=1)
                model.fit(Xtr, ytr)
                all_preds = np.vstack([t.predict(Xval_sel) for t in model.estimators_])
                mean_pred = np.mean(all_preds, axis=0)
                std_pred = np.std(all_preds, axis=0)
                abs_err = np.abs(yval_full - mean_pred)
                score_fold = adjusted_gower_normalization(abs_err, X_train_df, feat, num_cols, cat_cols, bool_cols)
                uncert_fold = std_pred

            preds_val += score_fold / self.n_splits
            uncert_val += uncert_fold / self.n_splits

        ap_single = average_precision_score(y_valid, preds_val)

        if progress_bar is not None:
            progress_bar.update(1)

        return i, feat, preds_val, uncert_val, ap_single

    # ---------------------------
    # Fit method
    # ---------------------------
    def fit(self, X_train, X_valid, y_valid, num_cols, cat_cols, bool_cols):
        start_time = time.time()
        np.random.seed(self.random_state)

        all_features = num_cols + cat_cols + bool_cols
        self.feature_indices_ = {f: i for i, f in enumerate(all_features)}

        # Preprocessing
        self.preprocessor_ = ColumnTransformer([
            ("num", RobustScaler(), num_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
            ("bool", "passthrough", bool_cols)
        ])

        Xtr_proc = self.preprocessor_.fit_transform(X_train)
        Xval_proc = self.preprocessor_.transform(X_valid)

        print(f"\n🚀 Step 1: RFOD with CV, feature bagging, and Gower normalization ({len(all_features)} features)...\n")

        # Train features in parallel
        #pbar = tqdm(total=len(all_features), desc="Training features", unit="feat")

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._train_one_feature)(
                i, feat, Xtr_proc, Xval_proc, self.feature_indices_, all_features,
                num_cols, cat_cols, bool_cols, y_valid, X_train, progress_bar=None
            )
            for i, feat in enumerate(all_features)
        )

        #pbar.close()
        #print(f"✅ Training completed in {(time.time()-start_time)/60:.2f} minutes.\n")

        # Collect results
        results.sort(key=lambda x: x[0])
        cell_scores = np.zeros((Xval_proc.shape[0], len(all_features)))
        cell_uncert = np.zeros((Xval_proc.shape[0], len(all_features)))
        feature_ap = []

        for i, feat, score, uncert, ap_single in results:
            cell_scores[:, i] = score
            cell_uncert[:, i] = uncert
            feature_ap.append((feat, ap_single))

        feature_ap = sorted(feature_ap, key=lambda x: x[1], reverse=True)
        ranked_feats = [f for f, _ in feature_ap]

        # Greedy subset selection
        selected, best_ap, no_improve = [], 0.0, 0
        progress = []

        print("\n⚙️ Greedy subset selection...\n")
        for feat in ranked_feats:
            temp_feats = selected + [feat]
            idxs = [self.feature_indices_[f] for f in temp_feats]
            weights = self._weight_from_uncertainty(cell_uncert[:, idxs], residual_mean=np.mean(cell_scores[:, idxs], axis=0))
            combined_score = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
            ap = average_precision_score(y_valid, combined_score)
            progress.append((feat, ap))

            if ap > best_ap:
                best_ap = ap
                selected.append(feat)
                no_improve = 0
                print(f"✅ Added: {feat:25s} | AP = {ap:.4f}")
            else:
                no_improve += 1
                print(f"⚠️ Rejected: {feat:25s} | AP = {ap:.4f}")

            if no_improve >= self.patience:
                print(f"\n⏹️ Early stopping (no improvement for {self.patience} rounds)\n")
                break

        # Final anomaly scores
        idxs = [self.feature_indices_[f] for f in selected]
        weights = self._weight_from_uncertainty(cell_uncert[:, idxs])
        final_combined = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
        self.final_AP_ = average_precision_score(y_valid, final_combined)
        self.ROC_AUC_ = roc_auc_score(y_valid, final_combined)
        self.selected_features_ = selected
        self.feature_ranking_ = pd.DataFrame(feature_ap, columns=["Feature", "SingleFeature_AP"])

        print(f"\n🏁 Final RFOD AP = {self.final_AP_:.4f}, ROC-AUC = {self.ROC_AUC_:.4f}")
        print("✅ Selected Features:", selected)

        progress_df = pd.DataFrame(progress, columns=["Feature", "AP"])
        feature_rank_df = pd.DataFrame(feature_ap, columns=["Feature", "SingleFeature_AP"])

        progress_df.to_csv('Data/progress.csv', index=False)
        feature_rank_df.to_csv('Data/feature_ranking.csv', index=False)

        return self

    # ---------------------------
    # Predict method
    # ---------------------------
    def predict(self, X_new):
        X_new_proc = self.preprocessor_.transform(X_new)
        X_sel = X_new_proc[:, [self.feature_indices_[f] for f in self.selected_features_]]
        # Aggregate anomaly score (simplified for unseen data)
        anomaly_score = np.mean(np.abs(X_sel - X_sel.mean(axis=0)), axis=1)
        return anomaly_score

    # ---------------------------
    # Save / Load
    # ---------------------------
    def save(self, path="rfod_model.joblib"):
        joblib.dump(self, path)
        print(f"💾 Model saved to {path}")

    @classmethod
    def load(cls, path="rfod_model.joblib"):
        print(f"📂 Loading model from {path}")
        return joblib.load(path)

In [12]:
rfod = RFOD(n_estimators=400, n_splits=8, feature_frac=0.8, patience=5, random_state=42)

rfod.fit(X_train, X_valid, y_valid["target"], num_cols, cat_cols, bool_cols)


🚀 Step 1: RFOD with CV, feature bagging, and Gower normalization (64 features)...


⚙️ Greedy subset selection...

✅ Added: pct_months_late           | AP = 0.3772
✅ Added: Early_repayment_ratio     | AP = 0.3849
✅ Added: HighInterestRate          | AP = 0.4232
⚠️ Rejected: longest_unchanged_UPB     | AP = 0.4143
✅ Added: UPB_autocorr              | AP = 0.4238
⚠️ Rejected: HighRiskCredit            | AP = 0.3577
⚠️ Rejected: UPB_rebound_count         | AP = 0.4186
⚠️ Rejected: HighDTI                   | AP = 0.3963
✅ Added: OriginalDTI               | AP = 0.4514
✅ Added: UPB_kurtosis              | AP = 0.4518
⚠️ Rejected: 12_13_UPB_Diff            | AP = 0.4483
⚠️ Rejected: NumberOfUnits             | AP = 0.4058
⚠️ Rejected: CompositeRiskScore        | AP = 0.4242
⚠️ Rejected: UPB_skew                  | AP = 0.4268
⚠️ Rejected: 11_12_UPB_Diff            | AP = 0.4477
⚠️ Rejected: 8_9_UPB_Diff              | AP = 0.4472
⚠️ Rejected: OriginalUPB               | AP = 0.4259
⚠️ Reje

In [13]:
rfod.save("rfod_model.joblib")

💾 Model saved to rfod_model.joblib


In [14]:
rfod_loaded = RFOD.load("rfod_model.joblib")

📂 Loading model from rfod_model.joblib


In [21]:
def scale_scores_to_unit(anomaly_scores):
    min_score = np.min(anomaly_scores)
    max_score = np.max(anomaly_scores)
    scaled_scores = (anomaly_scores - min_score) / (max_score - min_score + 1e-12)
    return scaled_scores

# Example usage:
anom_scores = (rfod_loaded.predict(X_test))
anom_scores_scaled = scale_scores_to_unit(anom_scores)

In [24]:
sub = pd.DataFrame()

sub['Id'] = df_test['Id']
sub['target'] = anom_scores_scaled

sub.to_csv('Data/sub_rfod1.csv', index=False)

In [83]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline

def greedy_isolation_forest_feature_selection(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    num_cols: list,
    cat_cols: list,
    bool_cols: list,
    contamination: float = 0.05,
    min_features: int = 5,
    patience: int = 3
):
    candidate_features = num_cols + cat_cols + bool_cols
    selected_features = []
    best_ap = 0
    rounds_no_improve = 0

    print("🚀 Starting Greedy Feature Selection (Isolation Forest)...")

    while len(candidate_features) > 0:
        ap_per_feature = {}
        for feature in candidate_features:
            current_features = selected_features + [feature]

            cur_num = [f for f in current_features if f in num_cols]
            cur_cat = [f for f in current_features if f in cat_cols]
            cur_bool = [f for f in current_features if f in bool_cols]

            preprocessor = ColumnTransformer(
                transformers=[
                    ("num", StandardScaler(), cur_num),
                    ("cat", Pipeline([
                        ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                        ("scaler", StandardScaler())
                    ]), cur_cat),
                    ("bool", StandardScaler(), cur_bool)
                ],
                remainder="drop"
            )

            pipe = Pipeline([
                ("prep", preprocessor),
                ("clf", IsolationForest(contamination=contamination, random_state=42))
            ])

            pipe.fit(X_train[current_features])

            scores = -pipe["clf"].decision_function(pipe["prep"].transform(X_valid[current_features]))
            ap = average_precision_score(y_valid, scores)
            ap_per_feature[feature] = ap

        best_feature = max(ap_per_feature, key=ap_per_feature.get)
        best_feature_ap = ap_per_feature[best_feature]

        if best_feature_ap > best_ap:
            selected_features.append(best_feature)
            candidate_features.remove(best_feature)
            best_ap = best_feature_ap
            rounds_no_improve = 0
            print(f"✅ Added feature: {best_feature} | AP = {best_ap:.4f}")
        else:
            rounds_no_improve += 1
            candidate_features.remove(best_feature)
            print(f"⚠️ Feature {best_feature} did not improve AP (AP={best_feature_ap:.4f})")

        if len(selected_features) >= min_features and rounds_no_improve >= patience:
            print("⚠️ Early stopping triggered")
            break

    print(f"\n🎯 Selected features ({len(selected_features)}): {selected_features}")
    print(f"Best AP achieved: {best_ap:.4f}")
    return selected_features, best_ap

selected_features, best_ap = greedy_isolation_forest_feature_selection(
    X_train=X_train,
    X_valid=X_valid,
    y_valid=y_valid["target"].values,
    num_cols=num_cols,
    cat_cols=cat_cols,
    bool_cols=bool_cols,
    contamination=0.1,
    min_features=2,
    patience=10
)

🚀 Starting Greedy Feature Selection (Isolation Forest)...
✅ Added feature: pct_months_late | AP = 0.3309
✅ Added feature: UPB_rebound_count | AP = 0.3798
✅ Added feature: longest_unchanged_UPB | AP = 0.3899
⚠️ Feature OriginalDTI_MissFLag did not improve AP (AP=0.3847)
⚠️ Feature HighRiskCredit did not improve AP (AP=0.3814)
⚠️ Feature CreditScore_MissFLag did not improve AP (AP=0.3808)
⚠️ Feature CreditScore did not improve AP (AP=0.3780)
⚠️ Feature HighLTV did not improve AP (AP=0.3498)
⚠️ Feature HighInterestRate did not improve AP (AP=0.3468)
⚠️ Feature HighDTI did not improve AP (AP=0.3409)
⚠️ Feature MI_Pct did not improve AP (AP=0.3342)
⚠️ Feature Early_repayment_ratio did not improve AP (AP=0.3269)
⚠️ Feature BalloonIndicator did not improve AP (AP=0.3264)
⚠️ Early stopping triggered

🎯 Selected features (3): ['pct_months_late', 'UPB_rebound_count', 'longest_unchanged_UPB']
Best AP achieved: 0.3899


In [86]:
feat = selected_features

X_train_sel = X_train[feat].copy()

from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

cur_num = [f for f in feat if f in num_cols]

preprocessor = ColumnTransformer(
                transformers=[
                    ("num", StandardScaler(), cur_num),
                ],
                remainder="drop"
            )

pipe = Pipeline([
                ("prep", preprocessor),
                ("clf", IsolationForest(contamination=0.1, random_state=42))
            ])

pipe.fit(X_train[feat])

scores = -pipe["clf"].decision_function(pipe["prep"].transform(X_valid[feat]))
min_v, max_v = np.min(scores), np.max(scores)
anom_score = (scores - min_v) / (max_v - min_v + 1e-12)
ap = average_precision_score(y_valid['target'], scores)

print("AP using RFOD-selected features + IsolationForest:", ap)

AP using RFOD-selected features + IsolationForest: 0.38987320787361485


In [73]:
raw_anom = -pipe["clf"].decision_function(pipe["prep"].transform(X_test[feat]))

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

submission = pd.DataFrame()

submission['Id'] = df_test['Id']
submission['target'] = anom_score

submission.to_csv('Data/submission2.csv', index=False)

In [88]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import KFold
from joblib import Parallel, delayed, dump, load

def adjusted_gower_normalization(abs_errors, X_train_df, feat, num_cols, cat_cols, bool_cols, alpha=0.05, eps=1e-8):
    if feat in num_cols:
        feature_vals = X_train_df[feat].values
        q_low = np.quantile(feature_vals, alpha)
        q_high = np.quantile(feature_vals, 1 - alpha)
        scale = max(q_high - q_low, eps)
        return np.abs(abs_errors) / scale
    elif feat in cat_cols or feat in bool_cols:
        # abs_errors here should be 1 - predicted probability of true class
        return abs_errors
    else:
        return abs_errors

class RFOD:
    def __init__(self,
                 n_estimators=200,
                 n_splits=5,
                 feature_frac=0.7,
                 patience=5,
                 n_jobs=-1,
                 agg_weight_method="adaptive",
                 random_state=42,
                 verbose=True):
        self.n_estimators = n_estimators
        self.n_splits = n_splits
        self.feature_frac = feature_frac
        self.patience = patience
        self.n_jobs = n_jobs
        self.agg_weight_method = agg_weight_method
        self.random_state = random_state
        self.verbose = verbose

        # Attributes to be set after fitting
        self.selected_features_ = None
        self.feature_ranking_ = None
        self.preprocessor_ = None
        self.feature_indices_ = None
        self.final_AP_ = None
        self.ROC_AUC_ = None
        self.feature_models_ = {}
        self.feature_model_inputs_ = {}

    def _weight_from_uncertainty(self, uncert, residual_mean=None, eps=1e-8):
        method = self.agg_weight_method
        if method == "inv":
            return 1.0 / (uncert + eps)
        elif method == "logodds":
            u = np.clip(uncert, eps, 1.0 - eps)
            return np.log((1.0 - u) / (u + eps))
        elif method == "adaptive":
            if residual_mean is None:
                residual_mean = np.mean(uncert, axis=0)
            return 1.0 / (uncert + 0.5 * residual_mean + eps)
        return np.ones_like(uncert)

    def _train_one_feature(self, i, feat, Xtr_proc, Xval_proc, feature_indices, all_features,
                           num_cols, cat_cols, bool_cols, y_valid, X_train_df):

        rng = np.random.default_rng(self.random_state + i)
        others = [f for f in all_features if f != feat]
        selected_features = rng.choice(others, size=max(1, int(len(others) * self.feature_frac)), replace=False)
        idxs_sel = [feature_indices[f] for f in selected_features]

        self.feature_model_inputs_[feat] = selected_features

        Xtr_sel = Xtr_proc[:, idxs_sel]
        Xval_sel = Xval_proc[:, idxs_sel]
        ytr_full = Xtr_proc[:, feature_indices[feat]]
        yval_full = Xval_proc[:, feature_indices[feat]]

        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        preds_val = np.zeros(len(yval_full))
        uncert_val = np.zeros(len(yval_full))

        # To store the model trained on entire training data (after CV)
        # For inference use
        model_full = None

        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(Xtr_sel)):
            Xtr, Xval = Xtr_sel[tr_idx], Xtr_sel[val_idx]
            ytr = ytr_full[tr_idx]

            if feat in cat_cols or feat in bool_cols:
                model = RandomForestClassifier(n_estimators=self.n_estimators,
                                               random_state=self.random_state + fold_idx,
                                               n_jobs=1)
                ytr_ = np.round(ytr).astype(int)
                model.fit(Xtr, ytr_)
                prob = model.predict_proba(Xval_sel)
                prob_max = np.max(prob, axis=1)
                prob_true = np.array([
                    prob[i, np.where(model.classes_ == np.round(yval_full[i]).astype(int))[0][0]]
                    if np.round(yval_full[i]).astype(int) in model.classes_ else 0
                    for i in range(len(yval_full))
                ])
                score_fold = 1.0 - prob_true
                uncert_fold = 1.0 - prob_max
            else:
                model = RandomForestRegressor(n_estimators=self.n_estimators,
                                              max_depth=6,
                                              random_state=self.random_state + fold_idx,
                                              n_jobs=1)
                model.fit(Xtr, ytr)
                all_preds = np.vstack([t.predict(Xval_sel) for t in model.estimators_])
                mean_pred = np.mean(all_preds, axis=0)
                std_pred = np.std(all_preds, axis=0)
                abs_err = np.abs(yval_full - mean_pred)
                score_fold = adjusted_gower_normalization(abs_err, X_train_df, feat, num_cols, cat_cols, bool_cols)
                uncert_fold = std_pred

            preds_val += score_fold / self.n_splits
            uncert_val += uncert_fold / self.n_splits

        # Fit model on full training data to use for inference
        if feat in cat_cols or feat in bool_cols:
            model_full = RandomForestClassifier(n_estimators=self.n_estimators,
                                                random_state=self.random_state,
                                                n_jobs=1)
            y_full_ = np.round(ytr_full).astype(int)
            model_full.fit(Xtr_sel, y_full_)
        else:
            model_full = RandomForestRegressor(n_estimators=self.n_estimators,
                                               max_depth=8,
                                               random_state=self.random_state,
                                               n_jobs=1)
            model_full.fit(Xtr_sel, ytr_full)

        ap_single = average_precision_score(y_valid, preds_val)

        return i, feat, preds_val, uncert_val, ap_single, model_full

    def fit(self, X_train, X_valid, y_valid, num_cols, cat_cols, bool_cols):
        start_time = time.time()
        np.random.seed(self.random_state)

        all_features = num_cols + cat_cols + bool_cols
        self.feature_indices_ = {f: i for i, f in enumerate(all_features)}

        self.preprocessor_ = ColumnTransformer([
            ("num", RobustScaler(), num_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
            ("bool", "passthrough", bool_cols)
        ])

        Xtr_proc = self.preprocessor_.fit_transform(X_train)
        Xval_proc = self.preprocessor_.transform(X_valid)

        if self.verbose:
            print(f"\n🚀 Step 1: RFOD with CV, feature bagging, and Gower normalization ({len(all_features)} features)...\n")

        self.feature_model_inputs_ = {}

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._train_one_feature)(
                i, feat, Xtr_proc, Xval_proc, self.feature_indices_, all_features,
                num_cols, cat_cols, bool_cols, y_valid, X_train
            )
            for i, feat in enumerate(all_features)
        )

        if self.verbose:
            print(f"✅ Training completed in {(time.time() - start_time) / 60:.2f} minutes.\n")

        results.sort(key=lambda x: x[0])
        cell_scores = np.zeros((Xval_proc.shape[0], len(all_features)))
        cell_uncert = np.zeros((Xval_proc.shape[0], len(all_features)))
        feature_ap = []

        # Store models for inference
        self.feature_models_ = {}

        for i, feat, score, uncert, ap_single, model_full in results:
            cell_scores[:, i] = score
            cell_uncert[:, i] = uncert
            feature_ap.append((feat, ap_single))
            self.feature_models_[feat] = model_full

        feature_ap = sorted(feature_ap, key=lambda x: x[1], reverse=True)
        ranked_feats = [f for f, _ in feature_ap]

        if self.verbose:
            print("\n⚙️ Greedy subset selection...\n")
        selected, best_ap, no_improve = [], 0.0, 0
        progress = []

        for feat in ranked_feats:
            temp_feats = selected + [feat]
            idxs = [self.feature_indices_[f] for f in temp_feats]
            weights = self._weight_from_uncertainty(cell_uncert[:, idxs], residual_mean=np.mean(cell_scores[:, idxs], axis=0))
            combined_score = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
            ap = average_precision_score(y_valid, combined_score)
            progress.append((feat, ap))
            if ap > best_ap:
                best_ap = ap
                selected.append(feat)
                no_improve = 0
                if self.verbose:
                    print(f"✅ Added: {feat:25s} | AP = {ap:.4f}")
            else:
                no_improve += 1
                if self.verbose:
                    print(f"⚠️ Rejected: {feat:25s} | AP = {ap:.4f}")
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"\n⏹️ Early stopping (no improvement for {self.patience} rounds)\n")
                break

        idxs = [self.feature_indices_[f] for f in selected]
        weights = self._weight_from_uncertainty(cell_uncert[:, idxs])
        final_combined = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
        self.final_AP_ = average_precision_score(y_valid, final_combined)
        self.ROC_AUC_ = roc_auc_score(y_valid, final_combined)
        self.selected_features_ = selected
        self.feature_ranking_ = pd.DataFrame(feature_ap, columns=["Feature", "SingleFeature_AP"])

        if self.verbose:
            print(f"\n🏁 Final RFOD AP = {self.final_AP_:.4f}, ROC-AUC = {self.ROC_AUC_:.4f}")
            print("✅ Selected Features:", selected)

        return self

    def predict(self, X_new):
        """Predict anomaly scores for new data using trained feature models."""
        X_new_proc = self.preprocessor_.transform(X_new)
        n_samples = X_new_proc.shape[0]
        n_features = len(self.selected_features_)

        # Prepare arrays for cell scores and uncertainties
        cell_scores = np.zeros((n_samples, n_features))
        cell_uncert = np.zeros((n_samples, n_features))

        # Reconstruct each feature using its trained forest and compute anomaly scores
        for i, feat in enumerate(self.selected_features_):
            model = self.feature_models_[feat]
            idx_feat = self.feature_indices_[feat]

            # Input features: all except current feature
            input_idxs = [idx for idx in range(X_new_proc.shape[1]) if idx != idx_feat]
            X_input = X_new_proc[:, input_idxs]

            y_true = X_new_proc[:, idx_feat]

            # Predict with all trees in ensemble
            all_preds = np.vstack([t.predict(X_input) for t in model.estimators_])
            mean_pred = np.mean(all_preds, axis=0)
            std_pred = np.std(all_preds, axis=0)

            # Calculate cell anomaly score using adjusted Gower's distance
            # Numerical features
            if feat in num_cols:
                agd_score = adjusted_gower_normalization(mean_pred - y_true, self.preprocessor_.transformers_[0][1].inverse_transform(X_new_proc[:, self.preprocessor_.transformers_[0][2]]), feat, num_cols, cat_cols, bool_cols)
            else:
                # For categorical features, probability of true class from model
                if hasattr(model, "predict_proba"):
                    proba = model.predict_proba(X_input)
                    classes = model.classes_
                    prob_true = np.array([
                        proba[i, np.where(classes == y_true[i])[0][0]] if y_true[i] in classes else 0
                        for i in range(n_samples)
                    ])
                    agd_score = 1.0 - prob_true
                else:
                    agd_score = np.abs(mean_pred - y_true)  # Fallback

            cell_scores[:, i] = agd_score
            cell_uncert[:, i] = std_pred

        # Aggregate cell-level scores to row-level scores using Uncertainty-Weighted Averaging
        weights = self._weight_from_uncertainty(cell_uncert)
        weighted_scores = np.sum(weights * cell_scores, axis=1) / (np.sum(weights, axis=1) + 1e-8)
        return weighted_scores

    def save(self, path="rfod_model.joblib"):
        dump(self, path)
        if self.verbose:
            print(f"💾 Model saved to {path}")

    @classmethod
    def load(cls, path="rfod_model.joblib"):
        if cls.verbose:
            print(f"📂 Loading model from {path}")
        return load(path)


In [89]:
rfod = RFOD(n_estimators=250, n_splits=7, feature_frac=0.6, patience=10, random_state=42)

rfod.fit(X_train, X_valid, y_valid["target"], num_cols, cat_cols, bool_cols)


🚀 Step 1: RFOD with CV, feature bagging, and Gower normalization (64 features)...



/Users/aviralgoyal/Desktop/NUS/Courses/CS5344_Big_Data_Technology/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


✅ Training completed in 72.79 minutes.


⚙️ Greedy subset selection...

✅ Added: pct_months_late           | AP = 0.3876
✅ Added: Early_repayment_ratio     | AP = 0.4113
✅ Added: HighInterestRate          | AP = 0.4281
⚠️ Rejected: longest_unchanged_UPB     | AP = 0.4261
⚠️ Rejected: HighRiskCredit            | AP = 0.3716
⚠️ Rejected: UPB_autocorr              | AP = 0.4147
⚠️ Rejected: HighDTI                   | AP = 0.4074
✅ Added: UPB_rebound_count         | AP = 0.4339
⚠️ Rejected: UPB_kurtosis              | AP = 0.4206
⚠️ Rejected: NumberOfUnits             | AP = 0.4191
⚠️ Rejected: 12_13_UPB_Diff            | AP = 0.4263
⚠️ Rejected: OriginalDTI               | AP = 0.3927
✅ Added: CompositeRiskScore        | AP = 0.4346
⚠️ Rejected: CreditScore_MissFLag      | AP = 0.3082
⚠️ Rejected: HighLTV                   | AP = 0.3245
⚠️ Rejected: UPB_skew                  | AP = 0.4213
⚠️ Rejected: 11_12_UPB_Diff            | AP = 0.4337
⚠️ Rejected: 8_9_UPB_Diff              | AP = 0

In [90]:
rfod.save('rfod_model2.joblib')

💾 Model saved to rfod_model2.joblib


In [144]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import KFold
from joblib import Parallel, delayed, dump, load

def adjusted_gower_normalization(abs_errors, X_train_df, feat, num_cols, cat_cols, bool_cols, alpha=0.05, eps=1e-8):
    if feat in num_cols:
        feature_vals = X_train_df[feat].values
        q_low = np.quantile(feature_vals, alpha)
        q_high = np.quantile(feature_vals, 1 - alpha)
        scale = max(q_high - q_low, eps)
        return np.abs(abs_errors) / scale
    elif feat in cat_cols or feat in bool_cols:
        # abs_errors here should be 1 - predicted probability of true class
        return abs_errors
    else:
        return abs_errors

class RFOD:
    def __init__(self,
                 n_estimators=200,
                 n_splits=5,
                 feature_frac=0.7,
                 patience=5,
                 n_jobs=-1,
                 agg_weight_method="adaptive",
                 random_state=42,
                 verbose=True):
        self.n_estimators = n_estimators
        self.n_splits = n_splits
        self.feature_frac = feature_frac
        self.patience = patience
        self.n_jobs = n_jobs
        self.agg_weight_method = agg_weight_method
        self.random_state = random_state
        self.verbose = verbose

        # Attributes to be set after fitting
        self.selected_features_ = None
        self.feature_ranking_ = None
        self.preprocessor_ = None
        self.feature_indices_ = None
        self.final_AP_ = None
        self.ROC_AUC_ = None
        # Store trained models per feature for inference
        self.feature_models_ = {}
        # Save input feature subsets for each feature model
        self.feature_model_inputs_ = {}
        self.X_train_original = None

    def _weight_from_uncertainty(self, uncert, residual_mean=None, eps=1e-8):
        method = self.agg_weight_method
        if method == "inv":
            return 1.0 / (uncert + eps)
        elif method == "logodds":
            u = np.clip(uncert, eps, 1.0 - eps)
            return np.log((1.0 - u) / (u + eps))
        elif method == "adaptive":
            if residual_mean is None:
                residual_mean = np.mean(uncert, axis=0)
            return 1.0 / (uncert + 0.5 * residual_mean + eps)
        return np.ones_like(uncert)

    def _train_one_feature(self, i, feat, Xtr_proc, Xval_proc, feature_indices, all_features,
                           num_cols, cat_cols, bool_cols, y_valid, X_train_df):

        rng = np.random.default_rng(self.random_state + i)
        others = [f for f in all_features if f != feat]
        selected_features = rng.choice(others, size=max(1, int(len(others) * self.feature_frac)), replace=False)
        idxs_sel = [feature_indices[f] for f in selected_features]

        Xtr_sel = Xtr_proc[:, idxs_sel]
        Xval_sel = Xval_proc[:, idxs_sel]
        ytr_full = Xtr_proc[:, feature_indices[feat]]
        yval_full = Xval_proc[:, feature_indices[feat]]

        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        preds_val = np.zeros(len(yval_full))
        uncert_val = np.zeros(len(yval_full))

        # To store the model trained on entire training data (after CV)
        # For inference use
        model_full = None

        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(Xtr_sel)):
            Xtr, Xval = Xtr_sel[tr_idx], Xtr_sel[val_idx]
            ytr = ytr_full[tr_idx]

            if feat in cat_cols or feat in bool_cols:
                model = RandomForestClassifier(n_estimators=self.n_estimators,
                                               random_state=self.random_state + fold_idx,
                                               n_jobs=1)
                ytr_ = np.round(ytr).astype(int)
                model.fit(Xtr, ytr_)
                prob = model.predict_proba(Xval_sel)
                prob_max = np.max(prob, axis=1)
                prob_true = np.array([
                    prob[i, np.where(model.classes_ == np.round(yval_full[i]).astype(int))[0][0]]
                    if np.round(yval_full[i]).astype(int) in model.classes_ else 0
                    for i in range(len(yval_full))
                ])
                score_fold = 1.0 - prob_true
                uncert_fold = 1.0 - prob_max
            else:
                model = RandomForestRegressor(n_estimators=self.n_estimators,
                                              max_depth=6,
                                              random_state=self.random_state + fold_idx,
                                              n_jobs=1)
                model.fit(Xtr, ytr)
                all_preds = np.vstack([t.predict(Xval_sel) for t in model.estimators_])
                mean_pred = np.mean(all_preds, axis=0)
                std_pred = np.std(all_preds, axis=0)
                abs_err = np.abs(yval_full - mean_pred)
                score_fold = adjusted_gower_normalization(abs_err, X_train_df, feat, num_cols, cat_cols, bool_cols)
                uncert_fold = std_pred

            preds_val += score_fold / self.n_splits
            uncert_val += uncert_fold / self.n_splits

        # Fit model on full training data to use for inference
        if feat in cat_cols or feat in bool_cols:
            model_full = RandomForestClassifier(n_estimators=self.n_estimators,
                                                random_state=self.random_state,
                                                n_jobs=1)
            y_full_ = np.round(ytr_full).astype(int)
            model_full.fit(Xtr_sel, y_full_)
        else:
            model_full = RandomForestRegressor(n_estimators=self.n_estimators,
                                               max_depth=8,
                                               random_state=self.random_state,
                                               n_jobs=1)
            model_full.fit(Xtr_sel, ytr_full)

        ap_single = average_precision_score(y_valid, preds_val)

        return i, feat, preds_val, uncert_val, ap_single, model_full, selected_features

    def fit(self, X_train, X_valid, y_valid, num_cols, cat_cols, bool_cols):
        start_time = time.time()
        np.random.seed(self.random_state)

        self.X_train_original = X_train.copy()
        all_features = num_cols + cat_cols + bool_cols
        self.feature_indices_ = {f: i for i, f in enumerate(all_features)}

        self.preprocessor_ = ColumnTransformer([
            ("num", RobustScaler(), num_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
            ("bool", "passthrough", bool_cols)
        ])

        Xtr_proc = self.preprocessor_.fit_transform(X_train)
        Xval_proc = self.preprocessor_.transform(X_valid)

        if self.verbose:
            print(f"\n🚀 Step 1: RFOD with CV, feature bagging, and Gower normalization ({len(all_features)} features)...\n")

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._train_one_feature)(
                i, feat, Xtr_proc, Xval_proc, self.feature_indices_, all_features,
                num_cols, cat_cols, bool_cols, y_valid, X_train
            )
            for i, feat in enumerate(all_features)
        )

        if self.verbose:
            print(f"✅ Training completed in {(time.time() - start_time) / 60:.2f} minutes.\n")

        results.sort(key=lambda x: x[0])
        cell_scores = np.zeros((Xval_proc.shape[0], len(all_features)))
        cell_uncert = np.zeros((Xval_proc.shape[0], len(all_features)))
        feature_ap = []

        # Store models for inference
        self.feature_models_ = {}
        self.feature_model_inputs_ = {}

        for i, feat, score, uncert, ap_single, model_full, selected_features in results:
            cell_scores[:, i] = score
            cell_uncert[:, i] = uncert
            feature_ap.append((feat, ap_single))
            self.feature_models_[feat] = model_full
            self.feature_model_inputs_[feat] = selected_features

        feature_ap = sorted(feature_ap, key=lambda x: x[1], reverse=True)
        ranked_feats = [f for f, _ in feature_ap]

        if self.verbose:
            print("\n⚙️ Greedy subset selection...\n")
        selected, best_ap, no_improve = [], 0.0, 0
        progress = []

        for feat in ranked_feats:
            temp_feats = selected + [feat]
            idxs = [self.feature_indices_[f] for f in temp_feats]
            weights = self._weight_from_uncertainty(cell_uncert[:, idxs], residual_mean=np.mean(cell_scores[:, idxs], axis=0))
            combined_score = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
            ap = average_precision_score(y_valid, combined_score)
            progress.append((feat, ap))
            if ap > best_ap:
                best_ap = ap
                selected.append(feat)
                no_improve = 0
                if self.verbose:
                    print(f"✅ Added: {feat:25s} | AP = {ap:.4f}")
            else:
                no_improve += 1
                if self.verbose:
                    print(f"⚠️ Rejected: {feat:25s} | AP = {ap:.4f}")
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"\n⏹️ Early stopping (no improvement for {self.patience} rounds)\n")
                break

        idxs = [self.feature_indices_[f] for f in selected]
        weights = self._weight_from_uncertainty(cell_uncert[:, idxs])
        final_combined = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
        self.final_AP_ = average_precision_score(y_valid, final_combined)
        self.ROC_AUC_ = roc_auc_score(y_valid, final_combined)
        self.selected_features_ = selected
        self.feature_ranking_ = pd.DataFrame(feature_ap, columns=["Feature", "SingleFeature_AP"])

        if self.verbose:
            print(f"\n🏁 Final RFOD AP = {self.final_AP_:.4f}, ROC-AUC = {self.ROC_AUC_:.4f}")
            print("✅ Selected Features:", selected)

        return self

    def predict(self, X_new):
        """Predict anomaly scores for new data using trained feature models."""
        X_new_proc = self.preprocessor_.transform(X_new)
        n_samples = X_new_proc.shape[0]
        n_features = len(self.selected_features_)

        cell_scores = np.zeros((n_samples, n_features))
        cell_uncert = np.zeros((n_samples, n_features))

        for i, feat in enumerate(self.selected_features_):
            model = self.feature_models_[feat]
            idx_feat = self.feature_indices_[feat]

            # Use saved input feature subset for this feature's model
            selected_features_input = self.feature_model_inputs_[feat]
            input_idxs = [self.feature_indices_[f] for f in selected_features_input]

            input_idxs = np.array(input_idxs, dtype=int)

            X_input = X_new_proc[:, input_idxs]

            y_true = X_new_proc[:, idx_feat]

            # Predict with all trees in ensemble
            all_preds = np.vstack([t.predict(X_input) for t in model.estimators_])
            mean_pred = np.mean(all_preds, axis=0)
            std_pred = np.std(all_preds, axis=0)

            if feat in self.feature_indices_ and feat in self.X_train_original.columns and feat in self.selected_features_:
                agd_score = adjusted_gower_normalization(
                    mean_pred - y_true,
                    self.X_train_original,
                    feat,
                    self.selected_features_, # restrict to working feature list
                    cat_cols,
                    bool_cols)
            elif feat in cat_cols or feat in bool_cols:
                if hasattr(model, "predict_proba"):
                    proba = model.predict_proba(X_input)
                    classes = model.classes_
                    prob_true = np.array([
                        proba[j, np.where(classes == y_true[j])[0][0]] if y_true[j] in classes else 0
                        for j in range(n_samples)
                    ])
                    agd_score = 1.0 - prob_true
                else:
                    agd_score = np.abs(mean_pred - y_true)
            else:
                agd_score = np.abs(mean_pred - y_true)

            cell_scores[:, i] = agd_score
            cell_uncert[:, i] = std_pred

        weights = self._weight_from_uncertainty(cell_uncert)
        weighted_scores = np.sum(weights * cell_scores, axis=1) / (np.sum(weights, axis=1) + 1e-8)
        return weighted_scores

    def save(self, path="rfod_model.joblib"):
        dump(self, path)
        if self.verbose:
            print(f"💾 Model saved to {path}")

    @classmethod
    def load(cls, path="rfod_model.joblib"):
        print(f"📂 Loading model from {path}")
        return load(path)


In [145]:
rfod = RFOD(n_estimators=250, n_splits=7, feature_frac=0.6, patience=10, random_state=42)

rfod.fit(X_train, X_valid, y_valid["target"], num_cols, cat_cols, bool_cols)


🚀 Step 1: RFOD with CV, feature bagging, and Gower normalization (64 features)...



/Users/aviralgoyal/Desktop/NUS/Courses/CS5344_Big_Data_Technology/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


✅ Training completed in 76.98 minutes.


⚙️ Greedy subset selection...

✅ Added: pct_months_late           | AP = 0.3876
✅ Added: Early_repayment_ratio     | AP = 0.4113
✅ Added: HighInterestRate          | AP = 0.4281
⚠️ Rejected: longest_unchanged_UPB     | AP = 0.4261
⚠️ Rejected: HighRiskCredit            | AP = 0.3716
⚠️ Rejected: UPB_autocorr              | AP = 0.4147
⚠️ Rejected: HighDTI                   | AP = 0.4074
✅ Added: UPB_rebound_count         | AP = 0.4339
⚠️ Rejected: UPB_kurtosis              | AP = 0.4206
⚠️ Rejected: NumberOfUnits             | AP = 0.4191
⚠️ Rejected: 12_13_UPB_Diff            | AP = 0.4263
⚠️ Rejected: OriginalDTI               | AP = 0.3927
✅ Added: CompositeRiskScore        | AP = 0.4346
⚠️ Rejected: CreditScore_MissFLag      | AP = 0.3082
⚠️ Rejected: HighLTV                   | AP = 0.3245
⚠️ Rejected: UPB_skew                  | AP = 0.4213
⚠️ Rejected: 11_12_UPB_Diff            | AP = 0.4337
⚠️ Rejected: 8_9_UPB_Diff              | AP = 0

In [146]:
rfod.save('rfod_model3.joblib')

💾 Model saved to rfod_model3.joblib


In [147]:
rfod_load = rfod.load('rfod_model3.joblib')

📂 Loading model from rfod_model3.joblib


In [148]:
anom = rfod_load.predict(X_valid)

Predicting feature: pct_months_late
Input feature subset: ['PropertyType' 'ProgramIndicator' 'OriginalUPB' 'UPB_autocorr'
 'OccupancyStatus' 'DebtServiceRatio' '3_4_UPB_Diff' 'LTV_mean' 'Channel'
 '4_5_UPB_Diff' 'CreditScore_DTI_Ratio' 'MSA_freq_enc' 'OriginalLTV_delta'
 '12_13_UPB_Diff' 'HighDTI' 'MI_Pct' '7_8_UPB_Diff' 'CreditScore'
 'OriginalLTV' 'OriginalDTI_MissFLag' 'Interest_rate_slope'
 'FirstTimeHomebuyerFlag' 'CreditScore_LTV_Ratio' 'OriginalLoanTerm'
 'avg_repayment_ratio' '9_10_UPB_Diff' 'UPB_rebound_count' 'LTV_slope'
 '1_2_UPB_Diff' 'NumberOfBorrowers' 'CreditScore_MissFLag' 'LTV_std'
 'SellerName_freq_enc' 'Interest_rate_std' 'OriginalInterestRate'
 '8_9_UPB_Diff' 'HighLTV']
Input indices: [50 52  3 27 48 40 11 30 49 12 38 43 42 20 61  1 15  0  4 58 35 55 37  6
 22 17 26 32  9  7 57 31 45 34  5 16 60]
X_new_proc type: <class 'numpy.ndarray'>
X_new_proc shape: (5370, 64)
Predicting feature: Early_repayment_ratio
Input feature subset: ['HighLTV' '4_5_UPB_Diff' 'Channel' '1

In [150]:
def scale_scores_to_unit(anomaly_scores):
    min_score = np.min(anomaly_scores)
    max_score = np.max(anomaly_scores)
    scaled_scores = (anomaly_scores - min_score) / (max_score - min_score + 1e-12)
    return scaled_scores

anom_scaled = scale_scores_to_unit(anom)
ap = average_precision_score(y_valid['target'], anom_scaled)

print("AP using RFOD:", ap)

AP using RFOD: 0.4364450585548466


In [153]:
anom_scores = (rfod_load.predict(X_test))
anom_scores_scaled = scale_scores_to_unit(anom_scores)

Predicting feature: pct_months_late
Input feature subset: ['PropertyType' 'ProgramIndicator' 'OriginalUPB' 'UPB_autocorr'
 'OccupancyStatus' 'DebtServiceRatio' '3_4_UPB_Diff' 'LTV_mean' 'Channel'
 '4_5_UPB_Diff' 'CreditScore_DTI_Ratio' 'MSA_freq_enc' 'OriginalLTV_delta'
 '12_13_UPB_Diff' 'HighDTI' 'MI_Pct' '7_8_UPB_Diff' 'CreditScore'
 'OriginalLTV' 'OriginalDTI_MissFLag' 'Interest_rate_slope'
 'FirstTimeHomebuyerFlag' 'CreditScore_LTV_Ratio' 'OriginalLoanTerm'
 'avg_repayment_ratio' '9_10_UPB_Diff' 'UPB_rebound_count' 'LTV_slope'
 '1_2_UPB_Diff' 'NumberOfBorrowers' 'CreditScore_MissFLag' 'LTV_std'
 'SellerName_freq_enc' 'Interest_rate_std' 'OriginalInterestRate'
 '8_9_UPB_Diff' 'HighLTV']
Input indices: [50 52  3 27 48 40 11 30 49 12 38 43 42 20 61  1 15  0  4 58 35 55 37  6
 22 17 26 32  9  7 57 31 45 34  5 16 60]
X_new_proc type: <class 'numpy.ndarray'>
X_new_proc shape: (13426, 64)
Predicting feature: Early_repayment_ratio
Input feature subset: ['HighLTV' '4_5_UPB_Diff' 'Channel' '

In [154]:
sub = pd.DataFrame()

sub['Id'] = df_test['Id']
sub['target'] = anom_scores_scaled

sub.to_csv('Data/sub_rfod2.csv', index=False)